Show probabilities space for classification of a plausible transmission pair. This is currently Figure B.5 in the SI

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import importlib
import transmission_functions as tf
importlib.reload(tf)
import time
import scipy as sp
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
import scienceplots
plt.style.use('science')
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import to_rgb

# Sensitivity analysis or not (this means shifting the generation interval by shift_sensitivity = -2)
sensitivity = 0

sensitivity_output_str_list = ['', '_sensitivity']
sensitivity_output_str = sensitivity_output_str_list[sensitivity]
shift_list = [0, -2]
shift_sensitivity = shift_list[sensitivity]
# weights = int(input('Include weights in matrix (input True or False): '))

reweight_zero = True
reweight_zero_str = ['', '_reweight_zero'][int(reweight_zero)]

low_sr = False

print('Sensitivity Analysis: ' + str(bool(sensitivity)))
print('Reweighting zero: ' + str(reweight_zero) )

# Convert sequences to strings
def convert_to_string(Seq):
    string = str(Seq).upper()
    return regex.sub(r'[^ACTG]', '-', string)

# Function for calculating (unweighted) adjacency matrix 
def adjacency_matrix(hamming_mat, daydiff_mat, idxs, nseqs):
    infection_adj_mat = np.zeros((nseqs, nseqs))
    for i in range(nseqs):
        for j in range(nseqs):
            infection_adj_mat[i, j] = ((int(hamming_mat[i, j]), daydiff_mat[i, j]) in idxs) * (daydiff_mat[i, j] >= 0)  #[(int(i), j) in valid_idxs for i, j in zip(*(df['hamming_distance'].values, df['Day_difference'].values))]

    np.fill_diagonal(infection_adj_mat, 0)
    return infection_adj_mat

# Function for calculating weighted adjacency matrix 
def weighted_adj_mat(hamming_mat, daydiff_mat, idxs, nseqs, prob_mat):
    weighted_mat = np.zeros((nseqs, nseqs))
    for i in range(nseqs):
        for j in range(nseqs):
            hdist = int(hamming_mat[i, j])
            ddiff = int(daydiff_mat[i, j])
            if (hdist, ddiff) in valid_idxs:
                weighted_mat[i, j] = prob_mat[hdist, ddiff] * (ddiff >= 0)  #[(int(i), j) in valid_idxs for i, j in zip(*(df['hamming_distance'].values, df['Day_difference'].values))]
    np.fill_diagonal(weighted_mat, 0)
    return weighted_mat

# Max number of days - 21 is a decent cutoff to make transmission >21 days unlikely
ndays = 20 + 1
# Max number of substitutions - 9 is a decent cutoff to make transmission >9 substitutions unlikely
nsubs = 9
# Substitution rate - subject to sensitivity!
sr = 0.063
days = np.arange(ndays)
subs = np.arange(nsubs)
prob_mat = np.zeros((ndays, nsubs+1))

for sub in range(nsubs+1):
    for d, day in enumerate(days):

        prob = tf.scenario1_probability(day, sub, sr, shift = shift_sensitivity)
        prob_mat[d, sub] = prob
    # prob_mats += [prob_mat]
prob_mat = prob_mat.T
if reweight_zero:
    prob_mat += 5e-6
    prob_mat /= np.sum(prob_mat)

def get_idxs(ndays = 20 + 1,
            nsubs = 9,
            sr = 1/11, 
            shift = 0):

    days = np.arange(ndays)
    subs = np.arange(nsubs)
    prob_mat = np.zeros((ndays, nsubs+1))
    for sub in range(nsubs+1):
        for d, day in enumerate(days):

            prob = tf.scenario1_probability(day, sub, sr)
            prob_mat[d, sub] = prob
        # prob_mats += [prob_mat]
    prob_mat = prob_mat.T
    time = np.arange(21)
    cprob = (np.cumsum(tf.gamma_probability_discrete(time)))
    p_cutoff = 0.95
    t_cutoff = time[np.argwhere(cprob > p_cutoff)[0][0]]
    prob_mat_cutoff = np.zeros_like(prob_mat)
    all_subs = np.arange(10)
    inhomogeneous = False
    tdiff1 = 6

    for sub in range(nsubs+1):

        for d, day in enumerate(days):
            cprob_sub = (np.sum(tf.substitution_probability(all_subs[:sub+1], d-shift, sr, inhomogeneous=inhomogeneous)))
            cprob_day = (np.sum(tf.gamma_probability_discrete(time[:d+1], shift = shift)))
            

            prob_mat_cutoff[sub, d] = (int(cprob_sub <= p_cutoff) * int(d <=t_cutoff))
    valid_idxs = [(0, 0)] + list(zip(*np.where(prob_mat_cutoff == 1)))
    return valid_idxs

In [ ]:

# Sensitivity analysis or not (this means shifting the generation interval by shift_sensitivity = -2)
sensitivity = 0

sensitivity_output_str_list = ['', '_sensitivity']
sensitivity_output_str = sensitivity_output_str_list[sensitivity]
shift_list = [0, -2]
shift_sensitivity = shift_list[sensitivity]
# weights = int(input('Include weights in matrix (input True or False): '))
if sensitivity:
    reweight_zero = False
    sensitivity_title = ' (Sensitivity)'
else:
    reweight_zero = reweight_zero
    sensitivity_title = ''

    
reweight_zero_str = ['', '_reweight_zero'][int(reweight_zero)]

# print('Weighted graph = ' + str(weights == 1))
print('Sensitivity Analysis: ' + str(bool(sensitivity)))
print('Reweighting zero: ' + str(reweight_zero) )

# Convert sequences to strings
def convert_to_string(Seq):
    string = str(Seq).upper()
    return regex.sub(r'[^ACTG]', '-', string)

# Function for calculating (unweighted) adjacency matrix 
def adjacency_matrix(hamming_mat, daydiff_mat, idxs, nseqs):
    infection_adj_mat = np.zeros((nseqs, nseqs))
    for i in range(nseqs):
        for j in range(nseqs):
            infection_adj_mat[i, j] = ((int(hamming_mat[i, j]), daydiff_mat[i, j]) in idxs) * (daydiff_mat[i, j] >= 0)  #[(int(i), j) in valid_idxs for i, j in zip(*(df['hamming_distance'].values, df['Day_difference'].values))]

    np.fill_diagonal(infection_adj_mat, 0)
    return infection_adj_mat

# Function for calculating weighted adjacency matrix 
def weighted_adj_mat(hamming_mat, daydiff_mat, idxs, nseqs, prob_mat):
    weighted_mat = np.zeros((nseqs, nseqs))
    for i in range(nseqs):
        for j in range(nseqs):
            hdist = int(hamming_mat[i, j])
            ddiff = int(daydiff_mat[i, j])
            if (hdist, ddiff) in valid_idxs:
                weighted_mat[i, j] = prob_mat[hdist, ddiff] * (ddiff >= 0)  #[(int(i), j) in valid_idxs for i, j in zip(*(df['hamming_distance'].values, df['Day_difference'].values))]
    np.fill_diagonal(weighted_mat, 0)
    return weighted_mat

# Max number of days - 21 is a decent cutoff to make transmission >21 days unlikely
ndays = 20 + 1
# Max number of substitutions - 9 is a decent cutoff to make transmission >9 substitutions unlikely
nsubs = 9
# Substitution rate - subject to sensitivity!
if low_sr == True:
    sr = 0.063
else:
    sr = 1/11
days = np.arange(ndays)
subs = np.arange(nsubs)
prob_mat = np.zeros((ndays, nsubs+1))

for sub in range(nsubs+1):
    for d, day in enumerate(days):

        prob = tf.scenario1_probability(day, sub, sr, shift = shift_sensitivity)
        prob_mat[d, sub] = prob
    # prob_mats += [prob_mat]
prob_mat = prob_mat.T
if reweight_zero:
    prob_mat += 5e-6
    prob_mat /= np.sum(prob_mat)

def get_idxs(ndays = 20 + 1,
            nsubs = 9,
            sr = sr, 
            shift = 0, 
            sensitivity = sensitivity):

    days = np.arange(ndays)
    subs = np.arange(nsubs)
    prob_mat = np.zeros((ndays, nsubs+1))
    for sub in range(nsubs+1):
        for d, day in enumerate(days):

            prob = tf.scenario1_probability(day, sub, sr)
            prob_mat[d, sub] = prob
        # prob_mats += [prob_mat]
    prob_mat = prob_mat.T
    time = np.arange(21)
    cprob = (np.cumsum(tf.gamma_probability_discrete(time)))
    p_cutoff = 0.95
    t_cutoff = time[np.argwhere(cprob > p_cutoff)[0][0]]
    prob_mat_cutoff = np.zeros_like(prob_mat)
    all_subs = np.arange(10)
    inhomogeneous = False
    tdiff1 = 6

    for sub in range(nsubs+1):

        for d, day in enumerate(days):
            if sensitivity:
                cprob_sub = (np.sum(tf.substitution_probability(all_subs[:sub+1], d-shift, sr, inhomogeneous=inhomogeneous)))
            else:
                cprob_sub = (np.sum(tf.substitution_probability(all_subs[:sub+1], d, sr, inhomogeneous=inhomogeneous)))
            prob_sub = (np.sum(tf.substitution_probability(all_subs[:sub+1], d-shift, sr, inhomogeneous=inhomogeneous)))
            cprob_day = (np.sum(tf.gamma_probability_discrete(time[:d+1], shift = shift)))
            

            prob_mat_cutoff[sub, d] = (int(cprob_sub <= p_cutoff) * int(d <=t_cutoff))
    valid_idxs = [(0, 0)] + list(zip(*np.where(prob_mat_cutoff == 1)))
    return valid_idxs

In [ ]:

prob_mat_plot_sensitivity = prob_mat.copy()
valid_idxs_sensitivity = get_idxs(shift = shift_sensitivity)



pd.DataFrame(prob_mat_plot_sensitivity)
for i in range(prob_mat_plot_sensitivity.shape[0]):
    for j in range(prob_mat_plot_sensitivity.shape[1]):
        if (i, j) not in valid_idxs_sensitivity:
            prob_mat_plot_sensitivity[i, j] = 0
prob_mat_plot_sensitivity = np.round(prob_mat_plot_sensitivity[:5, :15], 6)
prob_mat_plot_sensitivity_binary = prob_mat_plot_sensitivity.copy()
prob_mat_plot_sensitivity_binary[prob_mat_plot_sensitivity_binary> 0  ]=1
plt.figure(figsize = (14,6))
# cmp = sns.diverging_palette(145, 300, s=60, as_cmap=True)
colors = (to_rgb('lemonchiffon'), to_rgb('seagreen')) #firebrick
cmp = LinearSegmentedColormap.from_list('Custom', colors, len(colors))
ax = sns.heatmap(prob_mat_plot_sensitivity_binary, annot = prob_mat_plot_sensitivity, 
                 cmap = cmp, cbar = True, linewidths = .25, linecolor = 'lightgrey', alpha = 1)
ax.set_xlabel('Days Between Tests', fontsize = 14)
ax.set_ylabel('Genetic Distance Between Sequences', fontsize = 14)
ax.set_title('Probability of Observing Genetic and Testing Data in a Transmission Pair' + sensitivity_title, fontsize = 16)
colorbar = ax.collections[0].colorbar
colorbar.set_ticks([0.25, 0.75])

colorbar.set_ticklabels(['Reject Plausible Infector \n', 'Confirm Plausible Infector \n'], ha = 'center',
                        rotation = 270, rotation_mode = 'anchor', fontsize = 12)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../../Figures/p_plausible_pair' + sensitivity_output_str + '.pdf')